# Task 6 — Intentional failure (Scenario C: retries, notifications, repair & rerun)

**Purpose of this task in the demo:** fail on purpose, so you can present:

* **Task retries** — this task is configured with `max_retries = 2`: watch attempts 1 → 2 → 3 in the run UI before it finally fails.
* **Failure status** — the task shows red; the whole run is marked FAILED.
* **Email notification** — the job's `on_failure` notification (your email) fires.
* **Repair & Rerun** — fix the cause (set the job parameter `allow_success = true`), repair the run, and observe that **only the failed task re-runs**: ingest / validate / check_row_count / transform_publish keep their green results and are NOT re-executed.

**How it decides:** reads the job parameter `allow_success` (default `false` in the job). `false` → raise an exception → task fails.

In [0]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1 — Read the switch parameter
# ══════════════════════════════════════════════════════════════════════
# WHAT IS HAPPENING HERE:
#   * The task parameter allow_success = {{job.parameters.allow_success}}
#     references the JOB-LEVEL parameter, so you can flip it at run time
#     ("Run now > Run with different parameters", or the Repair dialog)
#     WITHOUT touching any code.

dbutils.widgets.text("allow_success", "false", "Set true to let this task succeed (used by Repair & Rerun)")
allow_success = dbutils.widgets.get("allow_success").strip().lower()

print(f"allow_success parameter received: '{allow_success}'")

In [0]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2 — Do some 'work', then deliberately fail
# ══════════════════════════════════════════════════════════════════════
# WHAT IS HAPPENING HERE:
#   * The print lines represent real work (visible in the task output
#     when you open the task run in the Jobs UI).
#   * If allow_success != 'true' we RAISE an exception — the standard way
#     a notebook task reports failure. The job then:
#       1. marks this task FAILED (attempt 1),
#       2. retries it twice (max_retries=2, 10s interval) — attempts 2, 3,
#       3. after the final attempt, marks the whole RUN failed and sends
#          the on_failure email notification.

print("fail_demo: starting simulated work...")
print("fail_demo: work finished, evaluating the switch...")

if allow_success != "true":
    raise RuntimeError(
        "💥 INTENTIONAL FAILURE (Scenario C) — this task was configured to fail "
        "so you can demo retries, the failed status, the on_failure email, and "
        "Repair & Rerun. To fix it: set the job parameter allow_success=true, "
        "then Repair run > Re-run failed tasks."
    )

print("✅ fail_demo SUCCEEDED — allow_success was 'true'. This green result is what you see after a Repair run.")

## Scenario C — presenter script (what to say and show)

1. **Trigger the failure:** run the job with defaults (`allow_success=false`, sample data). After `ingest` → `validate` → `check_row_count (TRUE)` → `transform_publish` succeed, `fail_demo` fails. The run page shows it **red, with 3 attempts** (1 original + 2 retries — `max_retries=2` on this task, with a 10-second retry interval).
2. **Show the retry evidence:** open the failed task — each attempt has its own row with timestamps and the same exception.
3. **Show the email:** the job's Notifications settings include your email on failure — check the inbox (also shows the duration-threshold/health settings next to it).
4. **Fix the cause WITHOUT re-ingesting:** in the failed run page click **Repair run**, set the run parameter `allow_success` to `true`, and run the repair. **Only `fail_demo` re-executes** — ingest / validate / check_row_count / transform_publish keep their green results and are NOT rerun.
5. **Point it out:** this is 'Repair & Rerun' — failed tasks re-run while successful upstream tasks remain valid, saving all the compute of the earlier steps.

*If your repair dialog does not expose parameter editing:* edit the job's `allow_success` default to `true` first, then use **Repair run > Rerun with latest settings** — same effect.